In [0]:
%sql
CREATE TABLE IF NOT EXISTS smart_meters.gold.dim_household (
    lclid STRING,
    acorn_group STRING
) USING DELTA;

MERGE INTO smart_meters.gold.dim_household AS target
USING smart_meters.silver.dim_household AS source
ON target.lclid = source.LCLid
WHEN MATCHED THEN 
  UPDATE SET target.acorn_group = CAST(source.Acorn_grouped AS VARCHAR(50))
WHEN NOT MATCHED THEN 
  INSERT (lclid, acorn_group) VALUES (source.LCLid, CAST(source.Acorn_grouped AS VARCHAR(50)));

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.gold.dim_weather (
    sk_weather STRING,
    summary STRING,
    thermal_stress_level STRING,
    temp_bin_10 STRING
) USING DELTA;

MERGE INTO smart_meters.gold.dim_weather AS target
USING (SELECT DISTINCT sk_weather, summary, thermal_stress_level, temp_bin_10 FROM smart_meters.silver.weather_enriched) AS source
ON target.sk_weather = source.sk_weather
WHEN NOT MATCHED THEN 
  INSERT (sk_weather, summary, thermal_stress_level, temp_bin_10) 
  VALUES (source.sk_weather, source.summary, source.thermal_stress_level, source.temp_bin_10);

  SELECT * FROM smart_meters.gold.dim_weather LIMIT 10

In [0]:
%sql
-- 1. The Gold Table owns the SCD2 columns!
CREATE TABLE IF NOT EXISTS smart_meters.gold.dim_tariffs (
    sk STRING, -- We will use UUIDs for surrogate keys
    id INT,
    type STRING,
    period STRING,
    price DOUBLE,
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN
) USING DELTA;

-- 2. Create the staging view using Natural Key (tariff_id)
CREATE OR REPLACE TEMP VIEW vw_staged_tariffs AS
SELECT 
    source.tariff_id AS merge_key, 
    source.* 
FROM smart_meters.silver.dim_tariffs source
UNION ALL
SELECT NULL AS merge_key, source.* FROM smart_meters.silver.dim_tariffs source
JOIN smart_meters.gold.dim_tariffs target ON source.tariff_id = target.id AND target.is_current = true
WHERE source.price <> target.price;

-- 3. The MERGE
MERGE INTO smart_meters.gold.dim_tariffs AS target
USING vw_staged_tariffs AS staged
ON target.id = staged.merge_key 

-- SCENARIO A: Close old tariff
WHEN MATCHED AND target.is_current = true AND target.price <> staged.price THEN
  UPDATE SET target.valid_to = CURRENT_DATE(), target.is_current = false

-- SCENARIO B: Insert new tariff (Generates the SK right here!)
WHEN NOT MATCHED THEN
  INSERT (
      sk, 
      id, 
      type, 
      period, 
      price, 
      valid_from, 
      valid_to, 
      is_current
  )
  VALUES (
      UUID(), -- Generates a unique SK string for the Data Warehouse!
      staged.tariff_id, 
      staged.type, 
      staged.period, 
      staged.price, 
      DATE '2012-01-01', -- Set to before fact data starts
      NULL, 
      true
  );

SELECT * FROM smart_meters.gold.dim_tariffs LIMIT 10

In [0]:
%sql

CREATE TABLE IF NOT EXISTS smart_meters.gold.dim_time (
    id BIGINT,
    year INT,
    month INT,
    day INT,
    hour INT,
    date_time TIMESTAMP,
    season STRING,
    holiday_type STRING
) USING DELTA;

MERGE INTO smart_meters.gold.dim_time AS target
USING smart_meters.silver.dim_time AS source
ON target.id = source.time_id
WHEN NOT MATCHED THEN
  INSERT (id, year, month, day, hour, date_time, season, holiday_type)
  VALUES (source.time_id, source.year, source.month, source.day, source.hour, source.date_time, source.season, source.holiday_type);

SELECT * FROM smart_meters.gold.dim_time LIMIT 10


In [0]:
%sql
CREATE TABLE IF NOT EXISTS smart_meters.gold.fact_consumption (
    energy_spent DOUBLE,
    money_spent DOUBLE,
    dim_tariff_sk STRING,
    dim_household_lclid STRING,
    dim_time_id BIGINT,
    sk_weather STRING
) USING DELTA;

-- Clear existing duplicates
TRUNCATE TABLE smart_meters.gold.fact_consumption;

MERGE INTO smart_meters.gold.fact_consumption AS target
USING (
    SELECT
        fc.energy_spent,
        (fc.energy_spent * tar.price) AS money_spent,
        tar.sk AS dim_tariff_sk,
        fc.dim_household_lclid,
        fc.time_id as dim_time_id,
        t.sk_weather as sk_weather
    FROM smart_meters.silver.fact_consumption fc
    
    INNER JOIN smart_meters.gold.dim_household hh
        ON fc.dim_household_lclid = hh.lclid

    INNER JOIN smart_meters.silver.dim_time t
        ON fc.time_id = t.time_id
        
    INNER JOIN smart_meters.gold.dim_weather w
        ON t.sk_weather = w.sk_weather

    INNER JOIN smart_meters.gold.dim_tariffs tar
        ON fc.tariff_id = tar.id
        AND (CAST(fc.day AS DATE) >= tar.valid_from)    
        AND (CAST(fc.day AS DATE) <= tar.valid_to OR tar.valid_to IS NULL)
) AS source
ON target.dim_household_lclid = source.dim_household_lclid 
   AND target.dim_time_id = source.dim_time_id

WHEN MATCHED THEN 
    UPDATE SET 
        target.energy_spent = source.energy_spent,
        target.money_spent = source.money_spent,
        target.dim_tariff_sk = source.dim_tariff_sk,
        target.sk_weather = source.sk_weather

WHEN NOT MATCHED THEN 
    INSERT (energy_spent, money_spent, dim_tariff_sk, dim_household_lclid, dim_time_id, sk_weather)
    VALUES (source.energy_spent, source.money_spent, source.dim_tariff_sk, source.dim_household_lclid, source.dim_time_id, source.sk_weather);

SELECT * FROM smart_meters.gold.fact_consumption LIMIT 10

In [0]:
%sql

SELECT 
    energy_spent,
    money_spent,
    dim_tariff_sk,
    dim_household_lclid,
    dim_time_id,
    sk_weather,
    COUNT(*) AS duplicate_count
FROM smart_meters.gold.fact_consumption
GROUP BY 
    energy_spent,
    money_spent,
    dim_tariff_sk,
    dim_household_lclid,
    dim_time_id,
    sk_weather
HAVING COUNT(*) > 1

In [0]:
%sql

SELECT 
    *,
    COUNT(*) AS duplicate_count
FROM smart_meters.silver.fact_consumption
GROUP BY 
    energy_spent,
    tariff_id,
    dim_household_lclid,
    time_id,
    day,
    hour
HAVING COUNT(*) > 1